### [Binary Forecasting Methods for Financial Time Series](http://medium.com/data-science-collective/comparing-binary-forecasting-methods-for-financial-time-seriesevaluating-cdf-based-trees-e95d2798e3f4)

> Evaluating CDF‑Based Trees, Calibrated GLMs, Genetic Sequence Models, and Traditional Benchmarks

In [ ]:
!pip install -q numpy pandas matplotlib scikit-learn plotly
!pip install -q scipy statsmodels yfinance
!pip install -q pygad xgboost lightgbm
!pip install -q torch prophet

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf

def fetch_spy(days=800):
    df = yf.Ticker("SPY").history(period=f"{days}d")[["Close"]].reset_index()
    df.rename(columns={"Date": "date", "Close": "close"}, inplace=True)
    return df

def label_event_onset(df, threshold=0.00):
    """
    Labels an event at time t if the maximum of the next two days' closes
    exceeds today's close by at least the given threshold.
    """
    # look one and two days ahead, take the elementwise max
    future_max = df["close"].shift(-1).combine(df["close"].shift(-2), max)
    df["event"] = (future_max > df["close"] * (1 + threshold)).astype(int)
    df.dropna(inplace=True)
    return df

# 2) FEATURE ENGINEERING (no leakage)
def make_features(df, n_lags=7, roll_windows=(5, 21)):
    for lag in range(1, n_lags + 1):
        df[f"lag_{lag}"] = df.close.pct_change(lag).shift(1)
    for w in roll_windows:
        df[f"roll_{w}"] = (
            df.close.pct_change()
              .rolling(w)
              .mean()
              .shift(1)
        )
    df.dropna(inplace=True)
    return df

#### Random Forest + Gaussian CDF Exceedance

Train a Random Forest to output a numerical score predicting our 0/1 event from past percent‑changes and rolling averages. Then measure how much those scores typically swing around the true labels to get a “margin of error.” For each new score, we imagine a bell curve centered on it and ask: “What’s the chance this curve lies above 0.5?” That gives us a probability of the event.


- **_Pro:_** Very quick to train, handles nonlinear interactions automatically.
- **_Con:_** Gaussian error assumption can mis‑calibrate probabilities if forest residuals are skewed or heteroskedastic.

In [ ]:
from sklearn.ensemble       import RandomForestRegressor
from sklearn.metrics        import log_loss, brier_score_loss, roc_auc_score
from sklearn.model_selection import TimeSeriesSplit

def benchmark_rf_cdf(train, test):
    Xtr = train.filter(like="lag_").join(train.filter(like="roll_"))
    ytr = train.event.values
    rf = RandomForestRegressor(n_estimators=200, random_state=42)
    rf.fit(Xtr, ytr)

    Xte = test.filter(like="lag_").join(test.filter(like="roll_"))
    pf = rf.predict(Xte)
    sigma = ytr.std()
    p1 = norm.sf(0.5, loc=pf, scale=sigma)

    return {
        "LogLoss": log_loss(test.event, p1),
        "Brier":   brier_score_loss(test.event, p1),
        "AUC":     roc_auc_score(test.event, p1)
    }, p1

#### Elastic‑Net Logistic Regression + Platt/Isotonic Calibration

Fit a logistic regression on lagged returns and rolling‑mean features (plus their pairwise interactions), using an elastic‑net penalty (a mix of L1 and L2) to automatically select and shrink coefficients. Then tune the amount of regularization via time‑series cross‐validation to maximize ROC. Those raw scores are then passed through a small calibrator — either a sigmoid fit (Platt scaling) or a monotonic piecewise mapping (isotonic) — so the final outputs are event probabilities.

- **_Pro:_** Produces sparse, interpretable models that are well‑calibrated after isotonic fitting.
- **_Con:_** Assumes a linear decision boundary in the expanded feature space — may miss complex, nonlinear temporal patterns.

In [ ]:
from sklearn.metrics        import log_loss, brier_score_loss, roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline       import Pipeline
from sklearn.preprocessing  import StandardScaler, PolynomialFeatures
from sklearn.linear_model   import LogisticRegressionCV
from sklearn.calibration    import CalibratedClassifierCV

def dynamic_baseline_full(df):

    # raw X,y
    X = df.filter(like="lag_").join(df.filter(like="roll_"))
    y = df.event.values

    # time‑series splits
    tscv = TimeSeriesSplit(n_splits=5)

    # build pipeline
    pipe = Pipeline([
        ("poly",   PolynomialFeatures(degree=2,
                                     interaction_only=True,
                                     include_bias=False)),
        ("scale",  StandardScaler()),
        ("lrcv",   LogisticRegressionCV(
                       Cs=[0.01, 0.1, 1.0, 10.0],
                       cv=tscv,
                       penalty="elasticnet",
                       solver="saga",
                       l1_ratios=[0.5, 0.7, 1.0],
                       class_weight="balanced",
                       scoring="roc_auc",
                       max_iter=2000,
                       n_jobs=-1,
                   ))
    ])

    # fit and report best hyperparams
    pipe.fit(X, y)
    best_lr = pipe.named_steps["lrcv"]
    print(f"[dynamic_baseline] best C = {best_lr.C_[0]:.3f}, l1_ratio = {best_lr.l1_ratio_[0]:.2f}, CV AUC ≈ {best_lr.scores_[1].mean():.4f}")

    # 5) isotonic calibration
    iso = CalibratedClassifierCV(pipe, method="isotonic", cv=tscv)
    iso.fit(X, y)

    # return calibrated probs on full set
    return iso.predict_proba(X)[:, 1]

#### GA–Evolved Softmax Binary Forecaster

Treat each lag/roll feature (and a bias) as a “gene” and use a genetic algorithm to evolve a weight vector that best separates 0/1 events. At each generation, candidate weight sets are scored by plugging into a logistic (sigmoid) p = σ(X·w) and measuring negative log‐loss on the training data. Through selection, crossover, and mutation, the GA hones in on a weight solution that minimizes classification error. The final evolved weights give a fast, simple linear forecaster with probabilities pₜ = σ(Xₜ·w).

- **_Pro:_** Searches non‑convex weight spaces; can discover bespoke linear combinations that classical solvers might miss.
- **_Con:_** Computationally expensive and stochastic; convergence not guaranteed, and raw logistic form can overfit if not regularized.

In [ ]:
import pygad
from scipy.stats import norm
from scipy.special import expit

def ga_binary_forecast(train, test,
                       pop_size=50, generations=100, mutation_percent=5):
    # build features & label arrays
    Xtr = train.filter(like="lag_").join(train.filter(like="roll_")).values
    ytr = train.event.values.astype(float)
    Xte = test.filter(like="lag_").join(test.filter(like="roll_")).values

    # append bias
    Xtr_ = np.hstack([Xtr, np.ones((len(Xtr),1))])
    Xte_ = np.hstack([Xte, np.ones((len(Xte),1))])
    n_genes = Xtr_.shape[1]

    # fitness must accept (ga, solution, sol_idx)
    def fitness_fn(ga_inst, solution, sol_idx):
        p = expit(Xtr_.dot(solution))
        # we return positive fitness = –log_loss
        return -log_loss(ytr, p)

    ga = pygad.GA(num_generations=generations,
                  num_parents_mating=pop_size//2,
                  fitness_func=fitness_fn,
                  sol_per_pop=pop_size,
                  num_genes=n_genes,
                  mutation_percent_genes=mutation_percent,
                  mutation_type="random",
                  crossover_type="single_point",
                  suppress_warnings=True)

    ga.run()
    w_opt, fitness, _ = ga.best_solution()
    print(f"[GA] best neg‑logloss = {fitness:.4f}")

    # test predictions
    p_te = expit(Xte_.dot(w_opt))
    return {
        "LogLoss": log_loss(test.event, p_te),
        "Brier":   brier_score_loss(test.event, p_te),
        "AUC":     roc_auc_score(test.event, p_te)
    }

#### XGBoost Classifier Benchmark

Similar to the RF+Gaussian‐CDF setup, train an XGBoost classifier on the same lagged and rolling features and use its built‑in `predict_proba` output as our event probability estimate.

- **_Pro:_** Often captures nonlinear interactions and outperforms RF in tabular tasks.
- **_Con:_** Can be overconfident; raw probabilities may require calibration (not done here).

In [ ]:
import xgboost as xgb
from xgboost import XGBClassifier
from sklearn.metrics import log_loss, brier_score_loss, roc_auc_score

def benchmark_xgb_clf(train, test):
    # Feature matrix & labels
    Xtr = train.filter(like="lag_").join(train.filter(like="roll_"))
    ytr = train.event.values
    Xte = test.filter(like="lag_").join(test.filter(like="roll_"))

    # Fit an XGBClassifier
    #    scale_pos_weight balances the positive vs negative class frequencies
    neg, pos = np.bincount(ytr)
    xgb_clf = XGBClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        scale_pos_weight=neg/pos,
        use_label_encoder=False,
        eval_metric="logloss",
        random_state=42
    )
    xgb_clf.fit(Xtr, ytr)

    # 3) Predict probabilities
    p1 = xgb_clf.predict_proba(Xte)[:,1]

    # 4) Compute standard metrics
    return {
        "LogLoss": log_loss(test.event, p1),
        "Brier":   brier_score_loss(test.event, p1),
        "AUC":     roc_auc_score(test.event, p1)
    }, p1

#### Markov Chain Model (State Space)

Treat “up‐move” vs. “no‐move” as two states. Simply count how often a non‐event today leads to an event tomorrow (P₀₁) and an event leads to another event (P₁₁). To forecast, look at yesterday’s actual state and use the matching transition probability. It’s simple and transparent but only remembers one day back.

- **_Pro:_** Extremely simple and ultra‑fast; captures first‑order persistence.
- **_Con:_** Ignores all continuous predictors; only models binary auto‐dependence.

In [ ]:
from sklearn.metrics import log_loss, brier_score_loss, roc_auc_score

def benchmark_markov(train, test):
    ev = train.event.values
    # count transitions
    c00 = ((ev[:-1]==0)&(ev[1:]==0)).sum()
    c01 = ((ev[:-1]==0)&(ev[1:]==1)).sum()
    c10 = ((ev[:-1]==1)&(ev[1:]==0)).sum()
    c11 = ((ev[:-1]==1)&(ev[1:]==1)).sum()
    p01 = c01 / (c00 + c01)  # P(1|0)
    p11 = c11 / (c10 + c11)  # P(1|1)

    # sequential one‐step‐ahead using true previous state
    prev = ev[-1]  # last train state
    p1 = []
    for actual in test.event.values:
        prob = p11 if prev==1 else p01
        p1.append(prob)
        prev = actual

    p1 = np.array(p1)
    return {
        "LogLoss": log_loss(test.event, p1),
        "Brier":   brier_score_loss(test.event, p1),
        "AUC":     roc_auc_score(test.event, p1)
    }, p1

#### ARIMA + Gaussian CDF Exceedance

Fit an ARIMA model, then for each new step forecast the mean and variance of the next return. Then compute the probability that the return exceeds the threshold by plugging into the Gaussian CDF. This links classic time‑series structure with a probabilistic rule.

- **_Pro:_** Leverages explicit time‐series structure and uncertainty modeling.
- **_Con:_** Refitting ARIMA at every step is slow; model on returns rather than directly on event dynamics can be suboptimal.

In [ ]:
import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.base.tsa_model import ValueWarning as StatsmodelsValueWarning

def arima_event_proba(train, test, threshold=0.005, order=(1,1,1)):
    """
    Fit an ARIMA on past returns and compute P(event=1) by
    approximating the one‐step ahead forecast error distribution.
    Suppresses statsmodels warnings about indexes & convergence.
    """
    # compute returns
    train_ret = train.close.pct_change().dropna()

    p1 = []
    for i in range(len(test)):
        # build history up to t–1
        history = pd.concat([train_ret, test.close.pct_change().iloc[:i]]).dropna()

        with warnings.catch_warnings():
            # ignore unsupported‐index & convergence warnings
            warnings.simplefilter("ignore", StatsmodelsValueWarning)
            warnings.simplefilter("ignore", ConvergenceWarning)

            model = ARIMA(history, order=order)
            fit   = model.fit()

            # one‐step forecast
            fc    = fit.get_forecast(steps=1)
            mu    = fc.predicted_mean.iloc[0]
            var   = fc.var_pred_mean.iloc[0]

        sigma = np.sqrt(var)
        # P(next_return > threshold)
        p1.append(1 - norm.cdf(threshold, loc=mu, scale=sigma))

    p1 = np.array(p1)
    mask = ~np.isnan(p1)
    return {
        "LogLoss": log_loss(test.event[mask], p1[mask]),
        "Brier":   brier_score_loss(test.event[mask], p1[mask]),
        "AUC":     roc_auc_score(test.event[mask], p1[mask])
    }, p1

#### Prophet + Gaussian CDF Exceedance

Fit _Facebook_’s Prophet model to past daily returns — capturing trend and weekly seasonality. Prophet gives us a point forecast (ŷ) and upper/lower uncertainty bounds. Then convert those bounds into a standard deviation σ, assume the next return is normally distributed, and then compute the probability of exceeding our threshold via the Gaussian tail (1–Φ((threshold–ŷ)/σ)). This blends trend fitting with a probabilistic rule.

- **_Pro:_** Captures seasonality and potential changepoints in returns.
- **_Con:_** Designed for continuous forecasting; using it for binary event thresholds is indirect.

In [ ]:
from prophet import Prophet
import numpy as np
import pandas as pd
from sklearn.metrics import log_loss, brier_score_loss, roc_auc_score
from scipy.stats import norm

def prophet_event_proba(train, test, threshold=0.003):

    # build returns history for Prophet
    dfp = train[['date','close']].rename(columns={'date':'ds','close':'y'})
    # compute simple pct‐returns
    dfp['y'] = dfp['y'].pct_change().fillna(0)
    # strip any timezone
    if pd.api.types.is_datetime64tz_dtype(dfp['ds']):
        dfp['ds'] = dfp['ds'].dt.tz_convert(None).dt.tz_localize(None)
    else:
        try:
            dfp['ds'] = dfp['ds'].dt.tz_localize(None)
        except (TypeError, ValueError):
            pass

    # fit Prophet on returns
    m = Prophet(
        daily_seasonality=False,
        weekly_seasonality=True,
        yearly_seasonality=False,
        changepoint_prior_scale=0.05
    )
    m.fit(dfp)

    # construct exactly the test‐set dates for forecasting
    future = test[['date']].rename(columns={'date':'ds'})
    # drop tz if any
    if pd.api.types.is_datetime64tz_dtype(future['ds']):
        future['ds'] = future['ds'].dt.tz_convert(None).dt.tz_localize(None)
    else:
        try:
            future['ds'] = future['ds'].dt.tz_localize(None)
        except (TypeError, ValueError):
            pass

    # predict returns on those dates
    fc = m.predict(future)
    yhat   = fc['yhat'].values
    sigma  = ((fc['yhat_upper'] - fc['yhat_lower'])/(2*1.96)).values
    sigma  = np.maximum(sigma, 1e-6)   # guard against zero width

    # compute P(return > threshold)
    p1 = norm.sf(threshold, loc=yhat, scale=sigma)
    p1 = np.clip(p1, 0, 1)

    # evaluate against the binary event in `test`
    #    note: test.event must already be defined consistently
    mask = ~np.isnan(p1)
    y_true = test['event'].to_numpy()[mask]
    y_pred = p1[mask]

    metrics = {
        "LogLoss": log_loss(y_true, y_pred),
        "Brier":   brier_score_loss(y_true, y_pred),
        "AUC":     max(1- roc_auc_score(y_true, y_pred), roc_auc_score(y_true, y_pred))
    }

    return metrics, p1

#### Recurrent Neural Network (LSTM)

Feed the RNN (LSTM) sequences of the last N days’ returns so it can learn temporal patterns and momentum effects. The LSTM layers process the time‑ordered data and output a single sigmoid score each day — probability that the event occurs next. We train with binary cross‑entropy and then slide the window forward at test time to generate event forecast.

- **_Pro:_** Learns temporal dependencies of arbitrary length, beyond fixed lags.
- **_Con:_** Requires careful tuning and larger datasets; may overfit on limited historical data.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

def rnn_event_proba(train, test, n_lags=7, hidden_dim=32,
                    num_layers=1,lr=1e-3, epochs=10, batch_size=32,device=None):
    """
    Train a simple LSTM on lagged returns to predict event[t], then
    output P(event=1) on `test`.
    """

    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # prepare return series + targets
    tr_ret = train.close.pct_change().fillna(0).values
    te_ret = test.close.pct_change().fillna(0).values
    tr_y   = train.event.values
    te_y   = test.event.values

    # build sequences
    X_tr, y_tr = [], []
    for i in range(n_lags, len(tr_ret)):
        X_tr.append(tr_ret[i-n_lags:i])
        y_tr.append(tr_y[i])
    X_te, y_te = [], []
    for i in range(n_lags, len(te_ret)):
        X_te.append(te_ret[i-n_lags:i])
        y_te.append(te_y[i])

    X_tr = torch.tensor(X_tr, dtype=torch.float32).unsqueeze(-1)
    y_tr = torch.tensor(y_tr, dtype=torch.float32)
    X_te = torch.tensor(X_te, dtype=torch.float32).unsqueeze(-1)
    y_te = torch.tensor(y_te, dtype=torch.float32)

    train_ds = TensorDataset(X_tr, y_tr)
    test_ds  = TensorDataset(X_te, y_te)
    tr_ld = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    te_ld = DataLoader(test_ds,  batch_size=batch_size)

    # LSTM/RNN
    class RNNForecast(nn.Module):
        def __init__(self, inp_dim, h_dim, nlayers):
            super().__init__()
            self.lstm = nn.LSTM(inp_dim, h_dim, nlayers, batch_first=True)
            self.fc   = nn.Linear(h_dim, 1)
        def forward(self, x):
            out, _ = self.lstm(x)
            h_last = out[:,-1,:]
            logit  = self.fc(h_last).squeeze(-1)
            return torch.sigmoid(logit)

    model = RNNForecast(1, hidden_dim, num_layers).to(device)
    opt   = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.BCELoss()

    # train
    model.train()
    for _ in range(epochs):
        for xb, yb in tr_ld:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            p = model(xb)
            loss_fn(p, yb).backward()
            opt.step()

    # predict
    model.eval()
    all_p = []
    with torch.no_grad():
        for xb, _ in te_ld:
            xb = xb.to(device)
            all_p.extend(model(xb).cpu().numpy())
    all_p = np.array(all_p)
    y_true = y_te.numpy()

    return {
        "LogLoss": log_loss(y_true, all_p),
        "Brier":   brier_score_loss(y_true, all_p),
        "AUC":     roc_auc_score(y_true, all_p)
    }, all_p

#### Attention‑Based Forecaster

Feed sliding windows of past returns into an attention network so the model can learn which specific lags are most relevant for predicting an up‑move. A linear layer projects each return into an embedding, then a multi‑head self‑attention block computes pairwise interactions across the window. Averaging the attended embeddings produces a fixed‑size summary, which a MLP with sigmoid activation turns into today’s event probability.

- **_Pro:_** Learns to weight past days adaptively and yields interpretable attention scores.
- **_Con:_** Higher compute cost and risk of overfitting on limited historical data.

In [ ]:
from sklearn.metrics import log_loss, brier_score_loss, roc_auc_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

def attention_event_proba(train, test, n_lags=7, embed_dim=32, num_heads=4,
                          hidden_dim=64, lr=1e-3, epochs=10, batch_size=32,
                          device=None):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # compute pct‐returns & align events
    tr_ret = train.close.pct_change().fillna(0).values
    te_ret = test.close.pct_change().fillna(0).values
    tr_evt = train.event.values
    te_evt = test.event.values

    # build sliding windows
    X_tr, y_tr = [], []
    for i in range(n_lags, len(tr_ret)):
        X_tr.append(tr_ret[i-n_lags:i])
        y_tr.append(tr_evt[i])
    X_te, y_te = [], []
    for i in range(n_lags, len(te_ret)):
        X_te.append(te_ret[i-n_lags:i])
        y_te.append(te_evt[i])

    X_tr = torch.tensor(X_tr, dtype=torch.float32).unsqueeze(-1)  # (N, L, 1)
    y_tr = torch.tensor(y_tr, dtype=torch.float32)
    X_te = torch.tensor(X_te, dtype=torch.float32).unsqueeze(-1)
    y_te = torch.tensor(y_te, dtype=torch.float32)

    tr_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=batch_size, shuffle=True)
    te_loader = DataLoader(TensorDataset(X_te, y_te), batch_size=batch_size)

    # model definition
    class AttnForecaster(nn.Module):
        def __init__(self, input_dim, embed_dim, num_heads, hidden_dim):
            super().__init__()
            self.project = nn.Linear(input_dim, embed_dim)
            self.attn    = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
            self.mlp     = nn.Sequential(
                nn.Linear(embed_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, 1)
            )

        def forward(self, x):
            # x: (B, L, 1)
            h = self.project(x)                  # → (B, L, E)
            a, _ = self.attn(h, h, h)            # → (B, L, E)
            v = a.mean(dim=1)                    # → (B, E)
            return torch.sigmoid(self.mlp(v)).squeeze(-1)  # → (B,)

    model = AttnForecaster(1, embed_dim, num_heads, hidden_dim).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    loss_fn   = nn.BCELoss()

    # training loop
    model.train()
    for _ in range(epochs):
        for xb, yb in tr_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            p = model(xb)
            loss_fn(p, yb).backward()
            optimizer.step()

    # prediction
    model.eval()
    preds = []
    with torch.no_grad():
        for xb, _ in te_loader:
            xb = xb.to(device)
            preds.extend(model(xb).cpu().numpy())

    y_true = y_te.numpy()
    p1     = np.array(preds)

    # metrics
    return {
        "LogLoss": log_loss(y_true, p1),
        "Brier":   brier_score_loss(y_true, p1),
        "AUC":     max(roc_auc_score(y_true, p1), 1- roc_auc_score(y_true, p1))
    }, p1

#### Run example

In [ ]:
# ——————————————————————————————
df = fetch_spy(700)
df = label_event_onset(df, threshold=0.003)
df = make_features(df)

split = int(len(df) * 0.8)
train_df, test_df = df.iloc[:split], df.iloc[split:]

bm_metrics, _ = benchmark_rf_cdf(train_df, test_df)
print("=== RF+Gaussian‑CDF BENCHMARK ===", bm_metrics)

dynp = dynamic_baseline_full(df)
print("=== Dynamic Logistic+Platt AUC (full) ===",
      roc_auc_score(df.event, dynp))


# 3) pure GA binary softmax forecaster
print("=== GA‑Softmax Binary Forecast AUC ===")
print(ga_binary_forecast(train_df, test_df))

print("\n=== XGB Classifier Benchmark ===")
print(benchmark_xgb_clf(train_df, test_df)[0])

# Markov Chain
mk_metrics, _ = benchmark_markov(train_df, test_df)
print("=== Markov Chain Benchmark ===", mk_metrics)

# Memba
# memb_metrics, _ = benchmark_hmm(train_df, test_df, n_states=2)
# print("=== Memba Model Benchmark ===", memb_metrics)

# arima for binary target
arima_m, _  = arima_event_proba(train_df, test_df)
print("=== ARIMA+CDF ===", arima_m)

prop_m, _   = prophet_event_proba(train_df, test_df)
print("=== Prophet+CDF ===", prop_m)

# RNN method
rnn_m, _ = rnn_event_proba(train_df, test_df,
                          n_lags=7,hidden_dim=32,
                          num_layers=1, lr=1e-3,
                          epochs=10, batch_size=32)
print("=== RNN Forecast AUC ===", rnn_m["AUC"])

attn_metrics, attn_p1 = attention_event_proba(train_df, test_df)
print("=== Attention Forecast AUC ===", attn_metrics)